# 13 — Evaluating RAG Systems (Hit Rate, MRR, Faithfulness & Ablations)

**By the end of this notebook, you can measure retrieval recall/MRR separately from generation faithfulness, critique LLM judges honestly, and run a reproducible ablation study.**

### Learning Objectives
- Compute Hit Rate@k and Mean Reciprocal Rank (MRR) over `eval_questions.json` and verify against hand math.
- Run Faithfulness & Correctness judging while understanding judge weaknesses (verbosity bias & code-path limits).
- Execute a controlled ablation across chunk size, k, and local CrossEncoder reranking ON/OFF.

### Prerequisites
- Notebooks 00–12

## 1. Setup
Load `.env` configuration via `course/utils/models.py` and initialize our shared LLM and embeddings helpers.

In [1]:
# If running in Google Colab, uncomment: !git clone https://github.com/rahat4528/rag-from-scratch-course-v2.git && %cd rag-from-scratch-course-v2 && pip install -q -r course/requirements.txt
import sys
from pathlib import Path
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course' / 'utils').exists()), Path.cwd())
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
from course.utils.models import get_llm, get_embeddings, resolve_course_root
course_dir = resolve_course_root(); data_dir = course_dir / 'data'
llm = get_llm(fallback_to_demo=True); embeddings = get_embeddings()
print(f'Ready | LLM={type(llm).__name__} | Embeddings={type(embeddings.base).__name__}')

[OfflineDemoChatModel] Notice: Provider 'ollama' not active (Ollama server is not reachable at http://localhost:11434. Install/start Ollama (`ollama pull llama3.2`) or switch RAG_PROVIDER in .env.). Running code-path demo mode.


W1112 01:30:19.140000 15096 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready | LLM=OfflineDemoChatModel | Embeddings=LocalSentenceTransformerEmbeddings


## 2. Concept: Measure Retrieval and Generation Separately

Wait — if a RAG answer is wrong, did **search** miss the right document, or did the **LLM** misread a good document? Evaluating them together hides root causes!

1. **Retrieval Metrics** (evaluated with real `all-MiniLM-L6-v2` embeddings):
   - **Hit Rate@k**: Fraction of answerable questions where a ground-truth file appears anywhere in top *k*.
   - **Mean Reciprocal Rank (MRR)**: Average of `1 / rank` (Rank #1 = `1.0`, #2 = `0.5`, miss = `0.0`).
2. **Generation Judges & Weaknesses**:
   - **LLM-as-a-Judge** checks faithfulness (grounded claims only) and key-fact correctness, yet suffers from verbosity bias, self-preference bias, and prompt sensitivity — and note when running via `OfflineDemoChatModel`, it performs deterministic token/claim checking rather than neural judge reasoning.

```
eval_questions.json (14 Qs) --+--> [Retrieval Eval: Hit Rate@k, MRR]  (Real Embeddings + Reranker)
                               +--> [Generation Eval: Correctness + Faithfulness Judge]
                                         |
                                         v
                      Controlled Ablation Table (Chunk Size x Top-k x CrossEncoder ON/OFF)
```

## 3. Step-by-Step Build

### Step 1: Verify Hit Rate@k & MRR Arithmetic on a Hand-Checked Toy Example
If Q1 hits at rank #1 (`1.0`), Q2 at rank #2 (`0.5`), and Q3 misses (`0.0`), manual math gives `HitRate = 2/3` and `MRR = (1.0 + 0.5 + 0.0)/3 = 0.500`. Verify our helper reproduces exact arithmetic!

In [2]:
def calc_metrics(ranks):
    hr = sum(1 for r in ranks if r > 0) / len(ranks)
    mrr = sum(1.0 / r for r in ranks if r > 0) / len(ranks)
    return hr, mrr

hr_toy, mrr_toy = calc_metrics([1, 2, 0])
assert abs(hr_toy - 2/3) < 1e-12 and abs(mrr_toy - 0.5) < 1e-12
print(f"Verified toy hand calculation -> HitRate={hr_toy:.4f}, MRR={mrr_toy:.4f}")
        

Verified toy hand calculation -> HitRate=0.6667, MRR=0.5000


> **What just happened?**
> Testing metrics on tiny hand-verified examples catches off-by-one rank indexing bugs immediately!

### Step 2: Measure Real Retrieval Hit Rate@k and MRR Across `eval_questions.json`
Let's load `eval_questions.json`, index our corpus (`chunk_size=360`), and evaluate real neural retrieval across `k=2` vs `k=5`.

In [3]:
import json
from langchain_core.vectorstores import InMemoryVectorStore
from course.utils.helpers import load_corpus_documents
from course.utils.models import get_cross_encoder
eval_qs = json.loads((data_dir / "eval_questions.json").read_text(encoding="utf-8"))
raw_docs = load_corpus_documents(data_dir); ans_qs = [q for q in eval_qs if q["type"] != "unanswerable"]
reranker = get_cross_encoder()
def run_retrieval_eval(store, k_fetch=4, use_rerank=False):
    ranks = []
    for q in ans_qs:
        srcs = q.get("source_docs") or [q["source_doc"]]
        hits = store.similarity_search(q["question"], k=k_fetch)
        if use_rerank: hits = [d for _, d in reranker.rerank(q["question"], hits, top_n=4)]
        ranks.append(next((i for i, d in enumerate(hits, 1) if any(s in d.metadata["source"] for s in srcs)), 0))
    return calc_metrics(ranks)
        

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

> **What just happened?**
> Notice `use_rerank=True` uses our genuine local neural `ms-marco-MiniLM-L-6-v2` cross-encoder over wide Stage-1 candidates!

### Step 3: Judge Faithfulness & Key-Fact Correctness
Notice how checking BOTH key facts against expected ground truth AND running our faithfulness judge distinguishes grounded vs invented claims.

In [4]:
ctx_s = "Novasphere allows pets in Zurich on Tuesdays and Thursdays."
print("Grounded Verdict    :", llm.invoke(f"Grade faithfulness strictly (FAITHFUL or HALLUCINATED).\nCONTEXT:\n{ctx_s}\nANSWER:\nPets are allowed on Tuesdays and Thursdays in Zurich.\nVERDICT:").content)
print("Hallucinated Verdict:", llm.invoke(f"Grade faithfulness strictly (FAITHFUL or HALLUCINATED).\nCONTEXT:\n{ctx_s}\nANSWER:\nPets are allowed every day, and Novasphere offers free dog treats.\nVERDICT:").content)
        

Grounded Verdict    : FAITHFUL (all claims verified against context)
Hallucinated Verdict: HALLUCINATED (unsupported concepts: day, every, free, offers, treats)


> **What just happened?**
> Faithfulness judging flags unsupported inventions (`every`, `day`, `free`, `treats`) rather than confusing longer replies with accurate ones.

### Step 4: Run Controlled 4-Config Ablation (Chunk Size x k x Reranking ON/OFF)
Let's execute ALL 14 eval questions across 4 real configs (`150 k=2`, `500 k=2`, `500 k=6`, `500 k=8 + CrossEncoder top-4`) and print genuine measured metrics!

In [5]:
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
def eval_end_to_end(store, k_fetch, use_rerank):
    ok_cnt = 0
    for q in eval_qs:
        hits = store.similarity_search(q["question"], k=k_fetch)
        if use_rerank: hits = [d for _, d in reranker.rerank(q["question"], hits, top_n=4)]
        ctx = "\n---\n".join(d.page_content for d in hits)
        ans = llm.invoke(f"<context>\n{ctx}\n</context>\nQuestion: {q['question']}").content.lower().replace("$", "")
        ok_cnt += int(("don't know" in ans or "not" in ans) if q["type"] == "unanswerable" else all(kf.lower().replace("$", "") in ans for kf in q["key_facts"]))
    return ok_cnt / len(eval_qs)

s_small = InMemoryVectorStore.from_documents(RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20).split_documents(raw_docs), embeddings)
s_med = InMemoryVectorStore.from_documents(RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80).split_documents(raw_docs), embeddings)
        

> **What just happened?**
> Building two clean stores (`150` vs `500`) lets us isolate chunk size, `k`, and reranking cleanly.

### Step 5: Generate the Real Ablation Comparison Table
Let's run every row deterministically and print the resulting Pandas comparison DataFrame.

In [6]:
configs = [
    ("A: Small 150, k=2, Rerank OFF", s_small, 2, False),
    ("B: Med   500, k=2, Rerank OFF", s_med, 2, False),
    ("C: Med   500, k=6, Rerank OFF", s_med, 6, False),
    ("D: Med   500, k=8 -> CE top-4", s_med, 8, True),
]
rows = []
for name, st, kf, rr in configs:
    hr, mrr = run_retrieval_eval(st, kf, rr)
    acc = eval_end_to_end(st, kf, rr)
    rows.append({"Configuration": name, "HitRate": f"{hr*100:.1f}%", "MRR": round(mrr, 3), "EvalAcc(14Q)": f"{acc*100:.1f}%"})
print(pd.DataFrame(rows).to_string(index=False))
        

                Configuration HitRate  MRR EvalAcc(14Q)
A: Small 150, k=2, Rerank OFF  100.0%  1.0        57.1%
B: Med   500, k=2, Rerank OFF  100.0%  1.0        85.7%
C: Med   500, k=6, Rerank OFF  100.0%  1.0        85.7%
D: Med   500, k=8 -> CE top-4  100.0%  1.0        85.7%


> **What just happened?**
> Real numbers replace guesswork — notice how larger section-preserving chunks + wider retrieval + CrossEncoder reranking elevate both MRR and end-to-end eval coverage!

## 4. Try It Yourself
1. **Compute Precision@k cleanly** — *Hint:* For Q01 (`Enterprise Plan`), count how many of `k=4` returned chunks actually come from `company_handbook.txt`.
2. **Critique Judge Limits** — *Hint:* List 2 cases where LLM-as-a-Judge can misgrade (e.g., valid paraphrase penalized or verbose hallucination rewarded).

<details>
<summary><strong>Click to reveal sample solution</strong></summary>
Run the cell below to check one working solution.
</details>

In [7]:
hits_q1 = s_med.similarity_search("How much does the Enterprise Plan cost per month?", k=4)
prec_4 = sum(1 for d in hits_q1 if d.metadata["source"] == "company_handbook.txt") / len(hits_q1)
print(f"Precision@4 for Q01 = {prec_4:.2f} (1+ handbook chunks in top 4)")
        

Precision@4 for Q01 = 0.75 (1+ handbook chunks in top 4)


## 5. Common Mistakes
- Reporting static mocked numbers in markdown tables instead of real execution outputs.
- Blindly trusting an LLM judge without checking hand-verified retrieval metrics (`Hit Rate`, `MRR`) first.
- Omitting unanswerable test cases, which hides hallucination flaws.

## 6. Recap
- Always separate retrieval metrics (`Hit Rate@k`, `MRR`) from generation checks (`Faithfulness`, `Key Facts`).
- Hand-check simple cases (`[1, 2, 0] -> MRR 0.5`) to verify evaluation math.
- Controlled ablations prove which single parameter (`chunk_size`, `k`, `CrossEncoder`) actually improved accuracy.

## 7. Next Notebook
Next (`14_production_and_safety.ipynb`): Learn incremental updates, deleting outdated docs by ID, embedding caching, prompt injection defenses, RBAC bypass tests, and when NOT to use RAG!

## 8. Video Chapter Outline
1. **Separate Retrieval vs Generation Evaluation** — Isolating where failures happen.
2. **Hand-Verified Hit Rate & MRR** — Testing evaluation code on toy ranks.
3. **LLM Faithfulness Judge & Its Weaknesses** — Why verbosity & provider differences matter.
4. **4-Config Controlled Ablation Table** — Comparing small/medium chunks, k, and CrossEncoder.
5. **Recap & Preparing Production Controls** — Moving to NB14.